# Models pipe

The project's models and their referee, in five steps. Every `predict_*` takes a StudyInstanceUID; every comparison is judged on the 58 human-labelled gold studies.

In [1]:
import sys; sys.path.insert(0, '..')   # repo root, so `functions` and `models` import

## 1 — David's model: images → labels

In [2]:
from functions.predict import predict_study

predict_study("1.2.826.0.1.3680043.8.498.10004873229099053869093324292195817260")

{'model_status': 'trained',
 'ACL': 0.5005950927734375,
 'MCL': 0.36429736018180847,
 'Medial Meniscus': 0.809837818145752,
 'Lateral Meniscus': 0.5134178996086121,
 'Medial OA': 0.7408567070960999,
 'Lateral OA': 0.6062192916870117,
 'PF OA': 0.6817634701728821,
 'Effusion': 0.7548481822013855,
 'Synovitis': 0.5048112273216248,
 "Baker's": 0.4826740026473999,
 'Contusion': 0.38908979296684265,
 'Fracture': 0.31344297528266907}

## 2 — David vs LLM
The image model against every LLM report reader. Prints a notice until the image checkpoint is trained.

In [3]:
from models.evaluate_labels import image_model_vs_llm, styled

t = image_model_vs_llm()
styled(t) if t is not None else t

,positives,llm_v4_blend,llm_full,llm_v2,pilkwang_v2,image_model
label,,,,,,
ACL,24.000,0.987,0.993,0.993,0.997,0.784
MCL,9.000,0.968,0.964,0.964,0.976,0.621
Medial Meniscus,26.000,0.948,0.954,0.954,0.943,0.709
Lateral Meniscus,23.000,0.879,0.862,0.862,0.841,0.804
Medial OA,15.000,0.932,0.931,0.931,0.908,0.901
Lateral OA,11.000,0.833,0.830,0.830,0.789,0.723
PF OA,21.000,0.902,0.901,0.901,0.891,0.773
Effusion,35.000,0.877,0.853,0.853,0.830,0.925
Synovitis,27.000,0.790,0.678,0.790,0.694,0.736


## 3 — Kevin's model: report → labels

In [4]:
from models.report_model import predict_report

predict_report("1.2.826.0.1.3680043.8.498.10004873229099053869093324292195817260")

{'ACL': 0.2281983196735382,
 'MCL': 0.2470674216747284,
 'Medial Meniscus': 0.9703903198242188,
 'Lateral Meniscus': 0.20251119136810303,
 'Medial OA': 0.9779312014579773,
 'Lateral OA': 0.20637544989585876,
 'PF OA': 0.2856284976005554,
 'Effusion': 0.9373799562454224,
 'Synovitis': 0.4119682013988495,
 "Baker's": 0.3022662401199341,
 'Contusion': 0.2402789443731308,
 'Fracture': 0.266244500875473}

## 4 — Kevin vs LLM
`kevin_rules` is the retired hardcoded baseline, kept as a data snapshot.

In [5]:
from models.evaluate_labels import report_model_vs_llm, styled

t = report_model_vs_llm()
styled(t) if t is not None else t

,positives,llm_v4_blend,llm_full,llm_v2,pilkwang_v2,report_model,kevin_rules
label,,,,,,,
ACL,24.000,0.987,0.993,0.993,0.997,0.913,0.817
MCL,9.000,0.968,0.964,0.964,0.976,0.909,0.863
Medial Meniscus,26.000,0.948,0.954,0.954,0.943,0.923,0.752
Lateral Meniscus,23.000,0.879,0.862,0.862,0.841,0.809,0.655
Medial OA,15.000,0.932,0.931,0.931,0.908,0.932,0.700
Lateral OA,11.000,0.833,0.830,0.830,0.789,0.830,0.514
PF OA,21.000,0.902,0.901,0.901,0.891,0.857,0.609
Effusion,35.000,0.877,0.853,0.853,0.830,0.652,0.547
Synovitis,27.000,0.790,0.678,0.790,0.694,0.779,0.621


## 5 — DAVID vs the best report reader
Images against whichever report-side reader (Kevin's model or an LLM table) scores best. `images_win` marks the findings where the pixels beat the report pipeline — the doctor-missed-it signal.

In [6]:
from models.evaluate_labels import david_vs_best_reader, styled

t = david_vs_best_reader()
styled(t) if t is not None else t

best report-side reader: llm_v4_blend


,positives,image_model,llm_v4_blend,images_win
label,,,,
ACL,24.000,0.784,0.987,False
MCL,9.000,0.621,0.968,False
Medial Meniscus,26.000,0.709,0.948,False
Lateral Meniscus,23.000,0.804,0.879,False
Medial OA,15.000,0.901,0.932,False
Lateral OA,11.000,0.723,0.833,False
PF OA,21.000,0.773,0.902,False
Effusion,35.000,0.925,0.877,True
Synovitis,27.000,0.736,0.790,False


## 6 — The full report
Every reader on one sheet: LLM readers, both image models (sagittal serving + multi-plane), Kevin's model, his rules baseline, and the **fusion model** (images + report, jointly trained — its column appears once `train_fusion` has run). Blue = beats every LLM reader on that finding.

In [7]:
from models.evaluate_labels import final_table, styled

styled(final_table())

,positives,llm_v4_blend,llm_full,llm_v2,pilkwang_v2,image_model,image_multiplane,report_model,kevin_rules
label,,,,,,,,,
ACL,24.000,0.987,0.993,0.993,0.997,0.784,0.817,0.913,0.817
MCL,9.000,0.968,0.964,0.964,0.976,0.621,0.590,0.909,0.863
Medial Meniscus,26.000,0.948,0.954,0.954,0.943,0.709,0.827,0.923,0.752
Lateral Meniscus,23.000,0.879,0.862,0.862,0.841,0.804,0.733,0.809,0.655
Medial OA,15.000,0.932,0.931,0.931,0.908,0.901,0.898,0.932,0.700
Lateral OA,11.000,0.833,0.830,0.830,0.789,0.723,0.776,0.830,0.514
PF OA,21.000,0.902,0.901,0.901,0.891,0.773,0.768,0.857,0.609
Effusion,35.000,0.877,0.853,0.853,0.830,0.925,0.932,0.652,0.547
Synovitis,27.000,0.790,0.678,0.790,0.694,0.736,0.694,0.779,0.621
